# 11-3절 연습 문제 풀이

이 노트북은 11-3절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch11/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 11-3절 공통 - 스타일 전이
from torchvision import models, transforms
from PIL import Image
IMAGE_SIZE = 256

def load_image(path, size=IMAGE_SIZE):
    img = Image.open(path).convert('RGB')
    tf = transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    return tf(img).unsqueeze(0).to(device)

class VGG19FeatureExtractor(nn.Module):
    def __init__(self, content_layer_idxs, style_layer_idxs):
        super().__init__()
        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        self.features = vgg.features
        self.content_layer_idxs = content_layer_idxs
        self.style_layer_idxs = style_layer_idxs
        for p in self.parameters(): p.requires_grad = False
    def forward(self, x):
        cf, sf = {}, {}
        for idx, layer in enumerate(self.features):
            x = layer(x)
            if idx in self.content_layer_idxs: cf[idx] = x
            if idx in self.style_layer_idxs: sf[idx] = x
        return cf, sf

def gram_matrix(feature):
    B, C, H, W = feature.shape
    f = feature.view(B, C, H * W)
    return torch.bmm(f, f.transpose(1, 2)) / (C * H * W)

mse_loss = nn.MSELoss()
def style_transfer(content_path, style_path, content_layers=[21],
                   style_layers=[0, 5, 10, 19, 28], content_weight=1.0,
                   style_weight=1e5, epochs=500, lr=0.03, style_size=IMAGE_SIZE):
    content_img = load_image(content_path)
    style_img = load_image(style_path, style_size)
    fx = VGG19FeatureExtractor(content_layers, style_layers).to(device).eval()
    with torch.no_grad():
        cf_ref, _ = fx(content_img)
        _, sf_ref = fx(style_img)
    generated = content_img.clone().requires_grad_(True)
    opt = torch.optim.Adam([generated], lr=lr)
    for e in range(epochs):
        cf, sf = fx(generated)
        c_loss = sum(mse_loss(cf[i], cf_ref[i]) for i in content_layers)
        s_loss = sum(mse_loss(gram_matrix(sf[i]), gram_matrix(sf_ref[i]))
                     for i in style_layers)
        loss = content_weight * c_loss + style_weight * s_loss
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad(): generated.clamp_(-3, 3)
    return generated.detach()

def show(img, title):
    mean = torch.tensor([0.485, 0.456, 0.406], device=img.device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=img.device).view(1, 3, 1, 1)
    viz.plot_images([(img * std + mean).clamp(0, 1)[0].cpu()], [title], images_per_row=1)

CONTENT = '../../data/Tuebingen_Neckarfront.jpg'
STYLE = '../../data/starry_night.jpg'

## 연습 11-10

예제에서 사용하지 않은 스타일 손실 가중치를 골라 가장 마음에 드는 결과를 찾아보자. 이때 콘텐츠 손실 가중치도 0.1, 1.0, 10.0 등으로 함께 바꿔 보면서 두 가중치의 비율이나 절댓값이 결과에 어떤 영향을 주는지 정리해 보자.

In [ ]:
for cw, sw in [(1.0, 1e4), (0.1, 1e5), (10.0, 1e5), (1.0, 1e6)]:
    out = style_transfer(CONTENT, STYLE, content_weight=cw, style_weight=sw, epochs=300)
    show(out, f'content={cw}, style={sw:.0e}')

결과를 좌우하는 것은 두 가중치의 **절댓값이 아니라 비율**이다. `content_weight=1, style_weight=1e5`와 `content_weight=10, style_weight=1e6`은 비율이 같아 거의 같은 결과를 낸다.

다만 절댓값이 함께 커지면 전체 손실이 커져 **학습률의 효과가 달라지므로**, 같은 에포크에서 수렴 정도가 달라질 수 있다.

## 연습 11-11

예제에서는 VGG-19 합성곱 블록에 포함된 16개 합성곱 계층 중 열 번째 계층에서 콘텐츠 특징을 추출한다. 콘텐츠 특징을 추출하는 합성곱 계층을 각각 첫 번째, 여섯 번째, 열여섯 번째 합성곱 계층으로 바꿔 생성 결과를 확인한 다음, 열 번째 합성곱 계층 선택의 적정성을 평가해 보자.

In [ ]:
# 인덱스: 첫 번째=0, 여섯 번째=12, 열 번째=21, 열여섯 번째=34
for name, idx in [('첫 번째(0)', 0), ('여섯 번째(12)', 12),
                  ('열 번째(21)', 21), ('열여섯 번째(34)', 34)]:
    out = style_transfer(CONTENT, STYLE, content_layers=[idx], epochs=300)
    show(out, f'콘텐츠 계층: {name}')

**얕은 계층**(0, 12)은 픽셀에 가까운 정보를 담아 원본이 거의 그대로 남고 스타일이 잘 입혀지지 않는다. **너무 깊은 계층**(34)은 추상적이라 형태가 크게 뭉개진다.

**열 번째(인덱스 21)** 는 사물의 배치와 구조는 유지하면서 질감은 자유롭게 바뀔 수 있는 지점이라, 스타일 전이에 가장 적절하다. 가티스 논문도 비슷한 깊이를 사용했다.

## 연습 11-12

스타일 특징은 특징의 규모에 따라서 미세한 붓 터치, 캔버스의 질감과 같은 작은 규모의 특징(고주파 특징)과 전반적인 색채 분위기나 큼직한 형태의 왜곡 패턴 같은 큰 규모의 특징(저주파 특징)으로 나눌 수 있다. 고주파 특징과 저주파 특징을 특정 범위(예를 들어 13x13 영역) 내에서만 관찰되는 특징과 그 범위보다 큰 특징으로 정의할 때, VGG-19 모델을 특징 추출기로 사용해 고주파 특징만 또는 저주파 특징만 전이하는 방법을 제시해 보자.

In [ ]:
# 얕은 계층 = 작은 수용 영역 = 고주파, 깊은 계층 = 저주파
print('고주파(작은 규모) 특징만 전이: 얕은 스타일 계층만 사용')
out_high = style_transfer(CONTENT, STYLE, style_layers=[0, 5], epochs=300)
show(out_high, '고주파 스타일만 (계층 0, 5)')

print('저주파(큰 규모) 특징만 전이: 깊은 스타일 계층만 사용')
out_low = style_transfer(CONTENT, STYLE, style_layers=[19, 28], epochs=300)
show(out_low, '저주파 스타일만 (계층 19, 28)')

**방법**: 스타일 손실에 사용할 계층을 골라 쓰면 된다.

- **고주파(13×13 이내의 작은 규모)**: 얕은 계층(0, 5)만 사용. 수용 영역이 작아 붓 터치와 캔버스 질감을 담는다.
- **저주파(그보다 큰 규모)**: 깊은 계층(19, 28)만 사용. 수용 영역이 넓어 색채 분위기와 큰 왜곡 패턴을 담는다.

VGG-19에서 인덱스 10 부근의 수용 영역이 대략 13×13이므로, 그 앞뒤로 나누면 문제의 정의와 맞는다.

## 연습 11-13

[도전 문제] 스타일 전이에서 스타일 이미지는 생성 이미지와 같은 크기일 필요가 없다. data 디렉터리에 저장된 <별이 빛나는 밤> 이미지는 1920x1520 크기인데, 본문 예제는 이를 256x256으로 변경해 사용한다. 스타일 이미지의 크기를 변환하지 말고 그대로 사용해 스타일 전이를 수행하고, 생성 결과에 어떤 차이가 있는지 확인해 보자.

In [ ]:
for size in (256, 1520):
    out = style_transfer(CONTENT, STYLE, style_size=size, epochs=300)
    show(out, f'스타일 이미지 크기 {size}')

그람 행렬은 (C, C) 크기라 **입력 크기와 무관**하다. 그래서 스타일 이미지를 원본 크기로 두어도 계산에는 문제가 없다.

달라지는 것은 **스타일 패턴의 상대적 크기**다. 큰 이미지를 그대로 쓰면 붓 터치 하나가 차지하는 픽셀 비율이 작아져, 생성 이미지에는 더 잘고 촘촘한 질감이 입혀진다. 반대로 스타일 이미지를 작게 줄이면 붓 터치가 굵고 크게 나타난다.

## 연습 11-14

[도전 문제] <별이 빛나는 밤>과 쌍벽을 이루는 표현주의 명작으로 에드바르 뭉크의 <절규>(data 디렉터리의 the_scream.jpg 파일)가 있다. <별이 빛나는 밤>과 <절규>의 스타일을 동시에 전이해 <별이 절규하는 튀빙겐 거리>를 만들어 보자.

11장 학습 노트

In [ ]:
import os
SCREAM = '../../data/the_scream.jpg'
if not os.path.exists(SCREAM):
    alt = '../../data/the_scream.jpg.jpg'
    if os.path.exists(alt):
        SCREAM = alt
        print(f'알림: 파일 이름이 the_scream.jpg.jpg 로 되어 있어 그대로 사용한다.')

# 두 스타일 이미지의 그람 행렬을 평균해 동시에 전이한다.
content_img = load_image(CONTENT)
style1, style2 = load_image(STYLE), load_image(SCREAM)
STYLE_LAYERS = [0, 5, 10, 19, 28]
fx = VGG19FeatureExtractor([21], STYLE_LAYERS).to(device).eval()
with torch.no_grad():
    cf_ref, _ = fx(content_img)
    _, sf1 = fx(style1)
    _, sf2 = fx(style2)
    gram_ref = {i: (gram_matrix(sf1[i]) + gram_matrix(sf2[i])) / 2
                for i in STYLE_LAYERS}

generated = content_img.clone().requires_grad_(True)
opt = torch.optim.Adam([generated], lr=0.03)
for e in range(500):
    cf, sf = fx(generated)
    c_loss = mse_loss(cf[21], cf_ref[21])
    s_loss = sum(mse_loss(gram_matrix(sf[i]), gram_ref[i]) for i in STYLE_LAYERS)
    loss = 1.0 * c_loss + 1e5 * s_loss
    opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): generated.clamp_(-3, 3)
show(generated.detach(), '별이 절규하는 튀빙겐 거리')

두 스타일을 섞는 가장 간단한 방법은 **각 계층의 그람 행렬을 평균**내어 목표로 삼는 것이다. 가중치를 주어 `0.7 * gram1 + 0.3 * gram2`처럼 한쪽을 강조할 수도 있다.

두 화풍이 모두 굵은 붓 터치와 소용돌이를 갖고 있어 자연스럽게 섞인다. 화풍 차이가 큰 그림을 섞으면 어느 쪽도 아닌 결과가 나오기 쉽다.